# Paper 3 reproducible analysis and numerical audit

This notebook reads the frozen daily-to-season pipeline outputs, verifies the acceptance gates and sample hierarchy, and independently recomputes one headline response. The full daily analysis is executed by `scripts/run_paper3.py`; no hidden notebook state is required.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np
import yaml

ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "paper3_enso.yaml").exists() and (ROOT.parent / "config" / "paper3_enso.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "config" / "paper3_enso.yaml").exists(), "Execute from paper3_execution or its notebooks directory"
sys.path.insert(0, str(ROOT / "src"))
from paper3core.statistics import observed_response

config = yaml.safe_load((ROOT / "config" / "paper3_enso.yaml").read_text(encoding="utf-8"))
OUTPUT = ROOT / "output"
config["study"]

{'area': 'Uttaradit',
 'country': 'Thailand',
 'analysis_period': [1981, 2014],
 'station_count_expected': 13}

## Acceptance gates and frozen provenance

In [2]:
gates = pd.read_csv(OUTPUT / "PAPER3_ACCEPTANCE_GATES.csv")
display(gates[["gate", "description", "status"]])
assert len(gates) == 8 and (gates["status"] == "PASS").all()

manifest = json.loads((OUTPUT / "paper3_run_manifest.json").read_text(encoding="utf-8"))
assert manifest["analysis_mode"] == "full_frozen"
manifest

,gate,description,status
0,P3-A,Observed and model ENSO sources are frozen and...,PASS
1,P3-B,Only complete common-365 station-seasons enter...,PASS
2,P3-C,All season ENSO labels use the frozen allowed ...,PASS
3,P3-D,Common period contains 13 gauges and seven models,PASS
4,P3-E,Blocked cross-fitted QDM used no target observ...,PASS
5,P3-F,Both phase-vs-neutral primary responses exist,PASS
6,P3-G,Phase contrast and true neutral-centred asymme...,PASS
7,P3-H,Pre-specified event-level bootstrap and permut...,PASS


{'config': 'C:\\MyPython\\AAA_cmip6_analysis_taylor diagram_dailyMonthly\\paper3_execution\\config\\paper3_enso.yaml',
 'config_sha256': '95809e909f887d9571331e243be632d6e38cc206a37c10cba71526ab9674a24c',
 'analysis_mode': 'full_frozen',
 'observed_oni_sha256': '0eacd96fe0e29f658f2ab190620f436d5c51294c190e0d082ef2f4b42c218768',
 'model_nino34_manifest_sha256': '27ce1dff3aa090c1ec2dc2c4894abc218feb048411dde1058635670b451ce76f',
 'hierarchy': 'daily -> management season -> station -> unique raw-grid signature -> model -> ensemble',
 'resampling_unit': 'management-season climate year; independently within model for model ensembles',
 'gates_passed': 8,
 'gates_total': 8}

## ENSO phase sample sizes

In [3]:
sample_sizes = pd.read_csv(OUTPUT / "enso_phase_sample_sizes.csv")
observed_samples = sample_sizes[sample_sizes["source_type"] == "OBSERVED"]
display(observed_samples)
assert observed_samples.groupby("season_type")["n_seasons"].sum().to_dict() == {"HOT_DRY": 33, "RAINY": 34}

,source_type,model,season_type,enso_phase,n_seasons
0,OBSERVED,OBSERVED,HOT_DRY,EL_NINO,8
1,OBSERVED,OBSERVED,HOT_DRY,LA_NINA,12
2,OBSERVED,OBSERVED,HOT_DRY,NEUTRAL,8
3,OBSERVED,OBSERVED,HOT_DRY,TRANSITION_UNCLASSIFIED,5
4,OBSERVED,OBSERVED,RAINY,EL_NINO,6
5,OBSERVED,OBSERVED,RAINY,LA_NINA,7
6,OBSERVED,OBSERVED,RAINY,NEUTRAL,8
7,OBSERVED,OBSERVED,RAINY,TRANSITION_UNCLASSIFIED,13


## Independent recomputation of a headline response

In [4]:
long = pd.read_csv(OUTPUT / "seasonal_metrics_long.csv.gz")
subset = long[
    (long["source_type"] == "OBSERVED")
    & (long["season_type"] == "RAINY")
    & (long["metric"] == "PRCPTOT")
]
recomputed, station_rows = observed_response(
    subset,
    phase="EL_NINO",
    minimum_seasons=config["enso"]["minimum_seasons_for_composite"],
)
saved = pd.read_csv(OUTPUT / "primary_response_summary.csv")
saved_value = saved.loc[
    (saved["source_type"] == "OBSERVED")
    & (saved["season_type"] == "RAINY")
    & (saved["metric"] == "PRCPTOT")
    & (saved["phase"] == "EL_NINO"),
    "response_pct",
].iloc[0]
print({"recomputed_pct": recomputed["response_pct"], "saved_pct": saved_value})
assert np.isclose(recomputed["response_pct"], saved_value, rtol=0, atol=1e-10)
display(station_rows.head())

{'recomputed_pct': -7.1655620496471935, 'saved_pct': np.float64(-7.165562049647206)}


,station,phase,n_phase,n_neutral,phase_mean,neutral_mean,response_absolute,response_pct,eligible
0,351001,EL_NINO,6,8,670.083333,789.9000,-119.816667,-15.168587,True
1,351002,EL_NINO,6,8,807.750000,849.6375,-41.887500,-4.930044,True
2,351003,EL_NINO,6,8,1084.450000,1107.7000,-23.250000,-2.098944,True
3,351004,EL_NINO,6,8,849.150000,855.4625,-6.312500,-0.737905,True
4,351005,EL_NINO,6,8,829.916667,893.9750,-64.058333,-7.165562,True


## Primary response, preservation, and asymmetry tables

In [5]:
responses = pd.read_csv(OUTPUT / "primary_response_summary.csv")
preservation = pd.read_csv(OUTPUT / "qdm_enso_signal_preservation.csv")
asymmetry = pd.read_csv(OUTPUT / "enso_asymmetry_primary.csv")
display(responses[["source_type", "season_type", "metric", "phase", "response_pct", "ci_low_pct", "ci_high_pct", "permutation_p_bh_primary"]])
display(preservation[["season_type", "metric", "phase", "category", "qdm_moved_toward_observed"]])
display(asymmetry[["source_type", "season_type", "metric", "phase_contrast_pct", "neutral_centered_asymmetry_pct"]])

,source_type,season_type,metric,phase,response_pct,ci_low_pct,ci_high_pct,permutation_p_bh_primary
0,OBSERVED,RAINY,PRCPTOT,EL_NINO,-7.165562,-18.760849,4.364156,0.470731
1,OBSERVED,RAINY,PRCPTOT,LA_NINA,2.721107,-10.214092,17.277570,0.789541
2,OBSERVED,RAINY,wet_day_frequency_pct,EL_NINO,-0.116959,-12.287937,2.820990,0.980000
3,OBSERVED,RAINY,wet_day_frequency_pct,LA_NINA,-2.355890,-9.583032,6.312917,0.646916
4,OBSERVED,RAINY,Rx1day,EL_NINO,-24.272931,-40.119865,3.047170,0.433440
5,OBSERVED,RAINY,Rx1day,LA_NINA,0.500455,-26.827212,35.802888,0.980000
6,OBSERVED,RAINY,CDD,EL_NINO,17.948718,-6.666667,40.954212,0.433440
7,OBSERVED,RAINY,CDD,LA_NINA,-5.828571,-19.049451,14.285714,0.711438
8,OBSERVED,HOT_DRY,PRCPTOT,EL_NINO,-28.704249,-55.479554,6.068544,0.433440
9,OBSERVED,HOT_DRY,PRCPTOT,LA_NINA,13.267463,-15.556308,57.988851,0.708655


,season_type,metric,phase,category,qdm_moved_toward_observed
0,RAINY,PRCPTOT,EL_NINO,PRESERVED,False
1,RAINY,PRCPTOT,LA_NINA,AMPLIFIED,False
2,RAINY,wet_day_frequency_pct,EL_NINO,INDETERMINATE_RAW_NEAR_ZERO,False
3,RAINY,wet_day_frequency_pct,LA_NINA,ATTENUATED,True
4,RAINY,Rx1day,EL_NINO,ATTENUATED,False
5,RAINY,Rx1day,LA_NINA,AMPLIFIED,False
6,RAINY,CDD,EL_NINO,PRESERVED,False
7,RAINY,CDD,LA_NINA,ATTENUATED,True
8,HOT_DRY,PRCPTOT,EL_NINO,REVERSED,False
9,HOT_DRY,PRCPTOT,LA_NINA,INDETERMINATE_RAW_NEAR_ZERO,True


,source_type,season_type,metric,phase_contrast_pct,neutral_centered_asymmetry_pct
0,OBSERVED,RAINY,PRCPTOT,11.120588,-3.485100
1,OBSERVED,RAINY,wet_day_frequency_pct,1.969494,-6.057715
2,OBSERVED,RAINY,Rx1day,14.769752,-22.959255
3,OBSERVED,RAINY,CDD,-19.390819,8.837409
4,OBSERVED,HOT_DRY,PRCPTOT,46.029102,-15.145910
5,OBSERVED,HOT_DRY,wet_day_frequency_pct,43.768997,15.476190
6,OBSERVED,HOT_DRY,Rx1day,20.496966,-23.896638
7,OBSERVED,HOT_DRY,CDD,-3.635546,-24.393883
8,RAW,RAINY,PRCPTOT,0.556289,-23.446829
9,RAW,RAINY,wet_day_frequency_pct,-0.455784,-12.166854


## Figure registry

In [6]:
figures = pd.read_csv(OUTPUT / "figures" / "Paper3_FIGURE_INDEX.csv")
display(figures)
assert len(figures) == 8 and (figures["bytes"] > 0).all()
assert figures["figure"].nunique() == 4

,figure,format,path,bytes
0,1,png,C:\MyPython\AAA_cmip6_analysis_taylor diagram_...,352394
1,1,pdf,C:\MyPython\AAA_cmip6_analysis_taylor diagram_...,44580
2,2,png,C:\MyPython\AAA_cmip6_analysis_taylor diagram_...,186229
3,2,pdf,C:\MyPython\AAA_cmip6_analysis_taylor diagram_...,39366
4,3,png,C:\MyPython\AAA_cmip6_analysis_taylor diagram_...,301311
5,3,pdf,C:\MyPython\AAA_cmip6_analysis_taylor diagram_...,45978
6,4,png,C:\MyPython\AAA_cmip6_analysis_taylor diagram_...,413332
7,4,pdf,C:\MyPython\AAA_cmip6_analysis_taylor diagram_...,49331
